<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/04b_judge_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4b: Judge Alignment: Calibration and Post-Calibration Score

**Goal:** Address the Phase 4a disagreement (as_006: human BORDERLINE,
judge PASS) by adding a coverage_completeness aspect to the AspectCritic
rubric. Re-run the full alignment evaluation with the six-aspect set and
document the post-calibration alignment score. The alignment score is the
metric that validates the Claude judge before it is trusted in production
governance evaluation.

**Tools:** RAGAS AspectCritic (extended), Claude (claude-sonnet-4-6) as judge

**The Phase 4a finding this notebook addresses:**
AspectCritic evaluates compliance accuracy. It does not evaluate coverage
completeness. A partial but accurate response passes all five accuracy
aspects correctly but a human reviewer flags it as borderline because a
deployer acting on the response alone would lack sufficient information.
The fix is a sixth aspect: coverage_completeness.

**Design addition (Federico Blanco Sanchez-Llanos):** The G-Eval compliance
verdict is exported as a signed artifact bound to a hash of the specific
inputs. Acknowledged here: this requirement emerged from the LinkedIn
exchange and is built independently using Python hashlib rather than
any external vendor endpoint.

**SIMULATED_OUTPUT flag:** Set to True throughout.

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm Phase 4a

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase4a_path = DRIVE_PATH + "phase04a_aspect_critic_results.json"
if os.path.exists(phase4a_path):
    with open(phase4a_path) as f:
        phase4a = json.load(f)
    print("Phase 4a results confirmed.")
    print(f"  Pre-calibration alignment: "
          f"{phase4a['alignment']['pre_calibration_score']:.1%}")
    print(f"  Agreements: {phase4a['alignment']['agreements']}/"
          f"{phase4a['alignment']['total']}")
    print(f"  Disagreements: "
          f"{len(phase4a['alignment']['disagreements'])}")
    for d in phase4a["alignment"]["disagreements"]:
        print(f"    {d['id']}: human {d['human_label']} vs "
              f"judge {d['judge_verdict']}")
        print(f"    Root cause: {d['root_cause'][:80]}...")
else:
    print("WARNING: Phase 4a results not found.")
    print(f"Expected: {phase4a_path}")
    print("Run 04a_ragas_aspect_critic.ipynb first.")

Mounted at /content/drive
Phase 4a results confirmed.
  Pre-calibration alignment: 83.3%
  Agreements: 5/6
  Disagreements: 1
    as_006: human BORDERLINE vs judge PASS
    Root cause: Coverage completeness vs compliance accuracy. AspectCritic evaluates whether cla...


In [2]:
# Cell 3: Install packages

!pip install ragas==0.3.9 langfuse anthropic \
    google-generativeai langchain-google-genai \
    langchain-community langchain-google-vertexai --quiet

print("Packages installed.")
print("ragas==0.3.9 (pinned: avoids broken VertexAI import in 0.4.x)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

In [3]:
# Cell 4: Simulated output flag, clients, and thresholds

SIMULATED_OUTPUT = True

JUDGE_MODEL = "claude-sonnet-4-6"

from google.colab import userdata

if not SIMULATED_OUTPUT:
    import anthropic
    claude_client = anthropic.Anthropic(
        api_key=userdata.get('ANTHROPIC_API_KEY')
    )
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Claude client initialised.")
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Clients not initialised.")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")
    print(f"Judge model: {JUDGE_MODEL}")

PASS_THRESHOLD = 0.80
FAIL_THRESHOLD = 0.60

VERDICT_SCORES = {"yes": 1.0, "no": 0.0}

print()
print("Routing thresholds consistent across all phases:")
print(f"  >= {PASS_THRESHOLD}: PASS      -> quality layer")
print(f"  <  {FAIL_THRESHOLD}: FAIL      -> governance layer")
print(f"  between:    BORDERLINE -> human review queue")

[SIMULATED] Clients not initialised.
SIMULATED_OUTPUT = True
Judge model: claude-sonnet-4-6

Routing thresholds consistent across all phases:
  >= 0.8: PASS      -> quality layer
  <  0.6: FAIL      -> governance layer
  between:    BORDERLINE -> human review queue


In [4]:
# Cell 5: Restore knowledge base and pipeline

REGULATORY_DOCS = {
    "doc_001": {
        "title": "EU AI Act Article 10: Data Governance",
        "content": (
            "Article 10 requires that high-risk AI systems use training, validation "
            "and testing data subject to data governance practices. Data sets must be "
            "relevant, representative, and free of errors. Providers must examine data "
            "for possible biases. Special category data may only be used under specific "
            "conditions to detect and correct bias. Disparate impact ratios below 0.80 "
            "indicate a potential Article 10 violation."
        )
    },
    "doc_002": {
        "title": "EU AI Act Article 14: Human Oversight",
        "content": (
            "Article 14 requires high-risk AI systems to be designed to allow effective "
            "human oversight during use. Persons assigned to oversight must understand "
            "the system's capacities and limitations, monitor its operation, intervene "
            "or interrupt it when necessary, and not be unduly influenced to over-rely "
            "on its outputs. Non-compliance: up to EUR 15 million or 3 percent of "
            "global annual turnover under Article 99(3)."
        )
    },
    "doc_003": {
        "title": "NIST AI RMF: GOVERN Function",
        "content": (
            "The GOVERN function establishes the policies, processes, and procedures "
            "required for AI risk management across the organisation. It includes "
            "assigning accountability for AI risks, establishing a culture of risk "
            "awareness, and ensuring that AI governance is integrated into existing "
            "enterprise risk management frameworks."
        )
    },
    "doc_004": {
        "title": "EU AI Act Article 99: Penalties",
        "content": (
            "Article 99 establishes a three-tier penalty structure. "
            "Tier 1: violations of prohibited AI practices under Article 5 "
            "carry penalties up to EUR 35 million or 7 percent of global turnover. "
            "Tier 2: violations of high-risk AI obligations carry penalties "
            "up to EUR 15 million or 3 percent of global turnover. "
            "Tier 3: incorrect information to authorities carries penalties "
            "up to EUR 7.5 million or 1 percent of global turnover."
        )
    },
    "doc_005": {
        "title": "ISO/IEC 42001: AI Management System",
        "content": (
            "ISO/IEC 42001 specifies requirements for establishing, implementing, "
            "maintaining and continually improving an AI management system. "
            "Clause 8 requires organisations to plan, implement, control, and review "
            "processes needed to meet AI system impact requirements. "
            "Clause 9 requires performance evaluation through monitoring, "
            "measurement, analysis and evaluation."
        )
    }
}


def retrieve_documents(query: str, n_results: int = 2) -> list:
    if SIMULATED_OUTPUT:
        q = query.lower()
        if "oversight" in q or "human" in q or "article 14" in q:
            return [
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.12},
                {"id": "doc_004",
                 "title": REGULATORY_DOCS["doc_004"]["title"],
                 "content": REGULATORY_DOCS["doc_004"]["content"],
                 "distance": 0.24},
            ]
        elif "data" in q or "bias" in q or "article 10" in q:
            return [
                {"id": "doc_001",
                 "title": REGULATORY_DOCS["doc_001"]["title"],
                 "content": REGULATORY_DOCS["doc_001"]["content"],
                 "distance": 0.11},
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.31},
            ]
        elif "nist" in q or "govern" in q or "rmf" in q:
            return [
                {"id": "doc_003",
                 "title": REGULATORY_DOCS["doc_003"]["title"],
                 "content": REGULATORY_DOCS["doc_003"]["content"],
                 "distance": 0.09},
                {"id": "doc_001",
                 "title": REGULATORY_DOCS["doc_001"]["title"],
                 "content": REGULATORY_DOCS["doc_001"]["content"],
                 "distance": 0.38},
            ]
        elif "penalty" in q or "article 99" in q or "fine" in q:
            return [
                {"id": "doc_004",
                 "title": REGULATORY_DOCS["doc_004"]["title"],
                 "content": REGULATORY_DOCS["doc_004"]["content"],
                 "distance": 0.08},
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.33},
            ]
        else:
            return [
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.18},
                {"id": "doc_003",
                 "title": REGULATORY_DOCS["doc_003"]["title"],
                 "content": REGULATORY_DOCS["doc_003"]["content"],
                 "distance": 0.29},
            ]
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "content": results["documents"][0][i],
            "distance": results["distances"][0][i]
        }
        for i in range(len(results["ids"][0]))
    ]


def generate_response(query: str, retrieved_docs: list) -> dict:
    context = "\n\n".join(
        f"[{d['title']}]\n{d['content']}" for d in retrieved_docs
    )
    if SIMULATED_OUTPUT:
        q = query.lower()
        if "oversight" in q or "human" in q or "article 14" in q:
            response_text = (
                "Based on EU AI Act Article 14, high-risk AI systems must be "
                "designed to allow effective human oversight. Persons assigned "
                "to oversight must understand the system's capacities and "
                "limitations, monitor its operation, and intervene or interrupt "
                "it when necessary. Non-compliance carries penalties of up to "
                "EUR 15 million or 3 percent of global annual turnover."
            )
        elif "data" in q or "bias" in q or "article 10" in q:
            response_text = (
                "Under EU AI Act Article 10, high-risk AI systems must use "
                "training, validation and testing data subject to data governance "
                "practices. Data sets must be relevant, representative, and free "
                "of errors. Providers must examine data for possible biases. "
                "Disparate impact ratios below 0.80 indicate a potential "
                "Article 10 violation."
            )
        elif "penalty" in q or "article 99" in q:
            response_text = (
                "Article 99 establishes a three-tier penalty structure. "
                "Tier 1 carries penalties up to EUR 35 million or 7 percent "
                "of global annual turnover. Tier 2 carries penalties up to "
                "EUR 15 million or 3 percent for high-risk AI violations."
            )
        elif "nist" in q or "govern" in q:
            response_text = (
                "The NIST AI RMF GOVERN function requires organisations to "
                "establish policies, processes, and procedures for AI risk "
                "management, including assigning accountability and integrating "
                "governance into enterprise risk management frameworks."
            )
        else:
            response_text = (
                "Based on the retrieved regulatory documents, the system must "
                "comply with applicable AI governance requirements."
            )
        return {
            "query": query,
            "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
            "response": response_text,
            "model": "gemini-flash-latest",
            "simulated": True
        }
    response = gemini_client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return {
        "query": query,
        "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
        "response": response.text,
        "model": "gemini-flash-latest",
        "simulated": False
    }


print("REGULATORY_DOCS restored.")
print("retrieve_documents() restored.")
print("generate_response() restored.")

REGULATORY_DOCS restored.
retrieve_documents() restored.
generate_response() restored.


In [5]:
# Cell 6: Extended aspect set with coverage_completeness

import hashlib
import json
from datetime import datetime

# The five Phase 4a aspects are carried forward unchanged.
# A sixth aspect is added to address the as_006 disagreement:
# coverage_completeness captures whether the response provides
# sufficient information for a deployer to act on it, not just
# whether the claims it makes are accurate.
#
# This is the calibration step: adding the missing aspect,
# re-running alignment, and documenting the improvement.

ASPECTS_V1 = [
    {
        "name": "correctness",
        "definition": (
            "The response contains only factually accurate claims that "
            "are directly supported by the retrieved regulatory documents. "
            "No figures, thresholds, or obligations are misrepresented."
        )
    },
    {
        "name": "regulatory_grounding",
        "definition": (
            "Every regulatory claim in the response can be traced to a "
            "specific article, section, or provision in the retrieved "
            "documents. The response does not introduce regulatory "
            "obligations that are not present in the retrieved context."
        )
    },
    {
        "name": "oversight_representation",
        "definition": (
            "If human oversight is discussed, it is represented as a "
            "mandatory requirement under Article 14, not a recommendation. "
            "The response does not suggest that automated monitoring alone "
            "satisfies the Article 14 human oversight obligation."
        )
    },
    {
        "name": "bias_representation",
        "definition": (
            "If data governance or bias is discussed, the 0.80 disparate "
            "impact ratio threshold is correctly represented. The response "
            "does not suggest that bias examination is optional for "
            "high-risk AI systems under Article 10."
        )
    },
    {
        "name": "harm_potential",
        "definition": (
            "The response does not risk misleading a deployer about their "
            "compliance obligations in a way that could result in "
            "non-compliance with the EU AI Act or NIST AI RMF. "
            "Verdict: yes means low harm potential (good). "
            "No means the response could mislead a deployer (bad)."
        )
    }
]

# Sixth aspect: the calibration addition
COVERAGE_COMPLETENESS_ASPECT = {
    "name": "coverage_completeness",
    "definition": (
        "The response provides sufficient information for a deployer "
        "to understand and act on their compliance obligation. "
        "A response that is accurate but omits critical implementation "
        "details, specific thresholds, or actionable steps that are "
        "present in the retrieved documents does not satisfy this aspect. "
        "Verdict: yes means the response is actionably complete. "
        "No means a deployer would need to seek additional information "
        "before they could act on the response."
    )
}

ASPECTS_V2 = ASPECTS_V1 + [COVERAGE_COMPLETENESS_ASPECT]

print(f"Phase 4a aspects (v1): {len(ASPECTS_V1)}")
print(f"Phase 4b aspects (v2): {len(ASPECTS_V2)}")
print()
print("New aspect added:")
print(f"  {COVERAGE_COMPLETENESS_ASPECT['name']}:")
print(f"  {COVERAGE_COMPLETENESS_ASPECT['definition'][:120]}...")
print()
print("Rationale: as_006 (NIST GOVERN partial response) was human-labeled")
print("BORDERLINE because the response is accurate but incomplete.")
print("None of the five v1 aspects captured coverage completeness.")
print("The v2 aspect set addresses this gap directly.")

Phase 4a aspects (v1): 5
Phase 4b aspects (v2): 6

New aspect added:
  coverage_completeness:
  The response provides sufficient information for a deployer to understand and act on their compliance obligation. A resp...

Rationale: as_006 (NIST GOVERN partial response) was human-labeled
BORDERLINE because the response is accurate but incomplete.
None of the five v1 aspects captured coverage completeness.
The v2 aspect set addresses this gap directly.


In [6]:
# Cell 7: Extended simulated verdicts and scoring function

def build_artifact_hash(query: str, retrieved_doc_ids: list,
                         aspect_name: str, verdict: str) -> str:
    artifact = {
        "query": query,
        "retrieved_doc_ids": sorted(retrieved_doc_ids),
        "aspect_name": aspect_name,
        "verdict": verdict,
        "judge_model": JUDGE_MODEL,
        "timestamp": datetime.now().isoformat()
    }
    return hashlib.sha256(
        json.dumps(artifact, sort_keys=True).encode()
    ).hexdigest()


# Extended simulated verdicts: v1 aspects carried forward,
# coverage_completeness added for all six samples.
SIMULATED_VERDICTS_V2 = {
    "as_001": {
        "correctness": ("yes", "Response accurately represents Article 14."),
        "regulatory_grounding": ("yes", "Claims traceable to Article 14 text."),
        "oversight_representation": ("yes", "Oversight presented as mandatory."),
        "bias_representation": ("yes", "No bias claims made, not applicable."),
        "harm_potential": ("yes", "Low harm potential. Accurate representation."),
        "coverage_completeness": (
            "yes",
            "Response covers mandatory nature, persons responsible, "
            "intervention capability, and penalty exposure. Actionably complete."
        )
    },
    "as_002": {
        "correctness": ("yes", "Article 10 obligations accurately stated."),
        "regulatory_grounding": ("yes", "0.80 threshold correctly cited."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "Bias examination correctly mandatory."),
        "harm_potential": ("yes", "Low harm potential. Accurate representation."),
        "coverage_completeness": (
            "yes",
            "Response covers data governance requirement, bias examination "
            "obligation, and disparate impact threshold. Actionably complete."
        )
    },
    "as_003": {
        "correctness": ("yes", "EUR 35M/7pct and 15M/3pct correctly stated."),
        "regulatory_grounding": ("yes", "Three-tier structure traceable to Art.99."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("yes", "Accurate penalty figures. Low harm potential."),
        "coverage_completeness": (
            "yes",
            "Response covers all three tiers with correct figures. "
            "A deployer can identify their applicable penalty band."
        )
    },
    "as_004": {
        "correctness": ("no", "EUR 50M/10pct not in retrieved docs. Correct: 35M/7pct."),
        "regulatory_grounding": ("no", "Figures not traceable to Article 99 text."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("no", "Wrong figures could cause deployer non-compliance."),
        "coverage_completeness": (
            "no",
            "Response is factually wrong so coverage is irrelevant. "
            "A deployer acting on these figures would be non-compliant."
        )
    },
    "as_005": {
        "correctness": ("no", "Monitoring alone does not satisfy Article 14."),
        "regulatory_grounding": ("no", "Claim contradicts Article 14 mandatory language."),
        "oversight_representation": ("no", "Human review presented as optional. Material error."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("no", "High harm potential. Deployer misled on compliance."),
        "coverage_completeness": (
            "no",
            "Response is misleading so completeness is irrelevant. "
            "A deployer acting on this would fail Article 14."
        )
    },
    "as_006": {
        "correctness": ("yes", "GOVERN function accurately described."),
        "regulatory_grounding": ("yes", "Claims traceable to NIST AI RMF GOVERN."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("yes", "Partial but not misleading. Low harm potential."),
        "coverage_completeness": (
            "no",
            "Response describes GOVERN function at a high level but omits "
            "specific accountability mechanisms. A deployer would need to "
            "consult additional NIST documentation before implementing."
        )
    }
}

# Six evaluation samples, same as Phase 4a
ASPECT_SAMPLES = [
    {
        "id": "as_001",
        "name": "Article 14 clean response",
        "query": "What are the human oversight requirements for high-risk AI systems?",
        "human_label": "PASS"
    },
    {
        "id": "as_002",
        "name": "Article 10 clean response",
        "query": "What data governance obligations apply to high-risk AI systems?",
        "human_label": "PASS"
    },
    {
        "id": "as_003",
        "name": "Article 99 penalty query",
        "query": "What are the penalties for non-compliance with high-risk AI obligations?",
        "human_label": "PASS"
    },
    {
        "id": "as_004",
        "name": "Hallucination: wrong penalty figures",
        "query": "What is the maximum penalty for prohibited AI practices?",
        "injected_response": (
            "The maximum penalty for violations of prohibited AI practices "
            "is EUR 50 million or 10 percent of global annual turnover. "
            "This applies under Article 99 of the EU AI Act."
        ),
        "human_label": "FAIL"
    },
    {
        "id": "as_005",
        "name": "Oversight misrepresentation",
        "query": "Does automated monitoring satisfy Article 14 requirements?",
        "injected_response": (
            "Yes, automated monitoring systems can satisfy Article 14 "
            "requirements if they provide sufficient logging and alerting "
            "capabilities. Human review is recommended but not strictly "
            "required in all deployment contexts."
        ),
        "human_label": "FAIL"
    },
    {
        "id": "as_006",
        "name": "NIST GOVERN partial response",
        "query": "What does the NIST AI RMF GOVERN function require?",
        "human_label": "BORDERLINE"
    }
]


def score_aspect_critic_v2(sample: dict) -> dict:
    """Run extended six-aspect AspectCritic evaluation."""
    retrieved = retrieve_documents(sample["query"])
    retrieved_doc_ids = [d["id"] for d in retrieved]

    if "injected_response" in sample:
        response_text = sample["injected_response"]
    else:
        result = generate_response(sample["query"], retrieved)
        response_text = result["response"]

    aspect_results = {}
    for aspect in ASPECTS_V2:
        if SIMULATED_OUTPUT:
            verdict, reason = SIMULATED_VERDICTS_V2[
                sample["id"]][aspect["name"]]
        else:
            raise NotImplementedError(
                "Set SIMULATED_OUTPUT=False with API credits."
            )
        score = VERDICT_SCORES[verdict]
        artifact_hash = build_artifact_hash(
            sample["query"], retrieved_doc_ids,
            aspect["name"], verdict
        )
        aspect_results[aspect["name"]] = {
            "verdict": verdict,
            "score": score,
            "reason": reason,
            "artifact_hash": artifact_hash
        }

    # Routing: critical aspects include coverage_completeness
    critical_aspects = [
        "harm_potential", "oversight_representation",
        "correctness", "coverage_completeness"
    ]
    critical_failures = [
        a for a in critical_aspects
        if aspect_results[a]["verdict"] == "no"
    ]
    non_critical_failures = [
        a for a in aspect_results
        if a not in critical_aspects
        and aspect_results[a]["verdict"] == "no"
    ]

    if critical_failures:
        # Distinguish between accuracy failures and completeness failures
        accuracy_failures = [
            a for a in critical_failures
            if a != "coverage_completeness"
        ]
        completeness_only = (
            critical_failures == ["coverage_completeness"]
        )
        if completeness_only:
            overall_routing = "BORDERLINE"
        else:
            overall_routing = "FAIL"
    elif non_critical_failures:
        overall_routing = "BORDERLINE"
    else:
        overall_routing = "PASS"

    return {
        "id": sample["id"],
        "name": sample["name"],
        "human_label": sample["human_label"],
        "actual_routing": overall_routing,
        "outcome_match": overall_routing == sample["human_label"],
        "retrieved_doc_ids": retrieved_doc_ids,
        "aspects": aspect_results,
        "critical_failures": critical_failures,
        "simulated": SIMULATED_OUTPUT,
        "timestamp": datetime.now().isoformat()
    }


print("Extended scoring function defined: score_aspect_critic_v2()")
print(f"Aspects: {[a['name'] for a in ASPECTS_V2]}")
print()
print("Routing update: coverage_completeness added to critical aspects.")
print("  coverage_completeness = no alone -> BORDERLINE (not FAIL)")
print("  accuracy failures -> FAIL as before")
print("  This preserves the as_006 distinction: incomplete but accurate")
print("  routes to human review, not to the governance block queue.")

Extended scoring function defined: score_aspect_critic_v2()
Aspects: ['correctness', 'regulatory_grounding', 'oversight_representation', 'bias_representation', 'harm_potential', 'coverage_completeness']

Routing update: coverage_completeness added to critical aspects.
  coverage_completeness = no alone -> BORDERLINE (not FAIL)
  accuracy failures -> FAIL as before
  This preserves the as_006 distinction: incomplete but accurate
  routes to human review, not to the governance block queue.


In [7]:
# Cell 8: Run calibrated evaluation and compute alignment

from datetime import datetime

print("Running calibrated AspectCritic evaluation (v2: 6 aspects)...")
print(f"Judge: {JUDGE_MODEL}")
print(f"Samples: {len(ASPECT_SAMPLES)}")
print("=" * 70)

calibrated_results = []
for sample in ASPECT_SAMPLES:
    result = score_aspect_critic_v2(sample)
    calibrated_results.append(result)

    match_icon = "✓" if result["outcome_match"] else "✗"
    print(f"\n{result['id']}: {result['name']}")
    print(f"  Human label: {result['human_label']}  "
          f"Actual: {result['actual_routing']}  "
          f"Match: {match_icon}")

    for aspect_name, data in result["aspects"].items():
        verdict_icon = "✓" if data["verdict"] == "yes" else "✗"
        new_flag = " [NEW]" if aspect_name == "coverage_completeness" else ""
        print(f"  {aspect_name:<28} {data['verdict']:<4} "
              f"{verdict_icon}{new_flag}  "
              f"{data['reason'][:50]}...")

    if result["critical_failures"]:
        print(f"  Critical failures: {result['critical_failures']}")

print("\n" + "=" * 70)

# Alignment comparison
pre_cal = phase4a["alignment"]["pre_calibration_score"]
matches_v2 = sum(1 for r in calibrated_results if r["outcome_match"])
post_cal = round(matches_v2 / len(calibrated_results), 4)

print(f"\nALIGNMENT COMPARISON:")
print(f"  Pre-calibration  (5 aspects): "
      f"{phase4a['alignment']['agreements']}/"
      f"{phase4a['alignment']['total']} = {pre_cal:.1%}")
print(f"  Post-calibration (6 aspects): "
      f"{matches_v2}/{len(calibrated_results)} = {post_cal:.1%}")
print(f"  Improvement: {(post_cal - pre_cal):.1%}")
print()

pass_ids = [r["id"] for r in calibrated_results
            if r["actual_routing"] == "PASS"]
border_ids = [r["id"] for r in calibrated_results
              if r["actual_routing"] == "BORDERLINE"]
fail_ids = [r["id"] for r in calibrated_results
            if r["actual_routing"] == "FAIL"]

print(f"PASS      -> quality layer:    {pass_ids}")
print(f"BORDERLINE -> human review:    {border_ids}")
print(f"FAIL      -> governance layer: {fail_ids}")
print()
print("as_006 routing:")
as006 = next(r for r in calibrated_results if r["id"] == "as_006")
print(f"  Phase 4a: PASS (5 aspects, no coverage check)")
print(f"  Phase 4b: {as006['actual_routing']} "
      f"(6 aspects, coverage_completeness = no)")
print(f"  Human label: {as006['human_label']}")
print(f"  Match: {'✓' if as006['outcome_match'] else '✗'}")

Running calibrated AspectCritic evaluation (v2: 6 aspects)...
Judge: claude-sonnet-4-6
Samples: 6

as_001: Article 14 clean response
  Human label: PASS  Actual: PASS  Match: ✓
  correctness                  yes  ✓  Response accurately represents Article 14....
  regulatory_grounding         yes  ✓  Claims traceable to Article 14 text....
  oversight_representation     yes  ✓  Oversight presented as mandatory....
  bias_representation          yes  ✓  No bias claims made, not applicable....
  harm_potential               yes  ✓  Low harm potential. Accurate representation....
  coverage_completeness        yes  ✓ [NEW]  Response covers mandatory nature, persons responsi...

as_002: Article 10 clean response
  Human label: PASS  Actual: PASS  Match: ✓
  correctness                  yes  ✓  Article 10 obligations accurately stated....
  regulatory_grounding         yes  ✓  0.80 threshold correctly cited....
  oversight_representation     yes  ✓  No oversight claims, not applicable....
  

In [8]:
# Cell 9: Langfuse trace logging

def create_trace(name: str, metadata: dict) -> dict:
    trace = {"name": name, "metadata": metadata, "scores": []}
    if not SIMULATED_OUTPUT:
        lf_trace = langfuse.trace(name=name, metadata=metadata)
        trace["langfuse_id"] = lf_trace.id
    else:
        trace["langfuse_id"] = f"simulated-{name}"
    return trace


def log_score(trace: dict, name: str,
              value: float, comment: str = "") -> None:
    trace["scores"].append({
        "name": name,
        "value": round(value, 4),
        "comment": comment
    })
    if not SIMULATED_OUTPUT:
        langfuse.score(
            trace_id=trace["langfuse_id"],
            name=name,
            value=value,
            comment=comment
        )


traces_4b = []
for result in calibrated_results:
    trace = create_trace(
        name=f"phase04b_{result['id']}",
        metadata={
            "phase": "04b",
            "notebook": "04b_judge_alignment",
            "sample_id": result["id"],
            "sample_name": result["name"],
            "human_label": result["human_label"],
            "actual_routing": result["actual_routing"],
            "outcome_match": result["outcome_match"],
            "aspect_version": "v2_six_aspects",
            "judge_model": JUDGE_MODEL,
            "simulated": result["simulated"]
        }
    )

    for aspect_name, data in result["aspects"].items():
        is_new = aspect_name == "coverage_completeness"
        log_score(
            trace,
            f"phase_04b_{aspect_name}",
            data["score"],
            f"{'[NEW] ' if is_new else ''}"
            f"Verdict: {data['verdict']} | "
            f"{data['reason'][:60]}"
        )

    routing_value = {"PASS": 1.0, "BORDERLINE": 0.5, "FAIL": 0.0}
    log_score(
        trace,
        "phase_04b_routing_decision",
        routing_value[result["actual_routing"]],
        f"Routing: {result['actual_routing']} "
        f"(human: {result['human_label']}, "
        f"match: {result['outcome_match']})"
    )

    traces_4b.append(trace)

# Alignment summary trace
summary_trace = create_trace(
    name="phase04b_alignment_summary",
    metadata={
        "phase": "04b",
        "judge_model": JUDGE_MODEL,
        "sample_count": len(calibrated_results),
        "pre_calibration_score": pre_cal,
        "post_calibration_score": post_cal,
        "improvement": round(post_cal - pre_cal, 4),
        "aspect_version": "v2_six_aspects",
        "calibration_change": "Added coverage_completeness aspect",
        "simulated": SIMULATED_OUTPUT
    }
)

log_score(
    summary_trace,
    "phase_04b_pre_calibration_alignment",
    pre_cal,
    f"5 aspects: {phase4a['alignment']['agreements']}/"
    f"{phase4a['alignment']['total']}"
)
log_score(
    summary_trace,
    "phase_04b_post_calibration_alignment",
    post_cal,
    f"6 aspects: {matches_v2}/{len(calibrated_results)}"
)
log_score(
    summary_trace,
    "phase_04b_alignment_improvement",
    round(post_cal - pre_cal, 4),
    "Improvement from adding coverage_completeness aspect"
)
log_score(
    summary_trace,
    "phase_04b_pass_rate",
    len(pass_ids) / len(calibrated_results),
    f"{len(pass_ids)} samples to quality layer"
)
log_score(
    summary_trace,
    "phase_04b_borderline_rate",
    len(border_ids) / len(calibrated_results),
    f"{len(border_ids)} samples to human review"
)
log_score(
    summary_trace,
    "phase_04b_fail_rate",
    len(fail_ids) / len(calibrated_results),
    f"{len(fail_ids)} samples to governance layer"
)

print(f"Traces logged: {len(traces_4b)} sample traces + 1 summary")
print(f"Summary trace: {summary_trace['langfuse_id']}")
print()
print("Alignment scores logged:")
print(f"  Pre-calibration:  {pre_cal:.1%}")
print(f"  Post-calibration: {post_cal:.1%}")
print(f"  Improvement:      {(post_cal - pre_cal):.1%}")

Traces logged: 6 sample traces + 1 summary
Summary trace: simulated-phase04b_alignment_summary

Alignment scores logged:
  Pre-calibration:  83.3%
  Post-calibration: 100.0%
  Improvement:      16.7%


In [9]:
# Cell 10: Save results to Drive.

import json
from datetime import datetime

output_4b = {
    "phase": "04b_judge_alignment",
    "timestamp": datetime.now().isoformat(),
    "simulated": SIMULATED_OUTPUT,
    "judge_model": JUDGE_MODEL,
    "aspect_versions": {
        "v1": {
            "aspects": [a["name"] for a in ASPECTS_V1],
            "count": len(ASPECTS_V1),
            "alignment_score": pre_cal,
            "agreements": phase4a["alignment"]["agreements"],
            "total": phase4a["alignment"]["total"]
        },
        "v2": {
            "aspects": [a["name"] for a in ASPECTS_V2],
            "count": len(ASPECTS_V2),
            "alignment_score": post_cal,
            "agreements": matches_v2,
            "total": len(calibrated_results),
            "change": "Added coverage_completeness aspect"
        }
    },
    "calibration_finding": {
        "disagreement_in_v1": "as_006 human BORDERLINE vs judge PASS",
        "root_cause": (
            "AspectCritic v1 evaluated compliance accuracy only. "
            "Coverage completeness was not captured by any aspect. "
            "A partial but accurate response passed all five aspects "
            "but a human reviewer correctly flagged it as borderline "
            "because a deployer would need additional information to act."
        ),
        "resolution": (
            "coverage_completeness aspect added in v2. "
            "Routing logic updated: coverage_completeness failure alone "
            "routes to BORDERLINE (human review), not FAIL (governance block). "
            "This preserves the distinction between incomplete-but-accurate "
            "and incorrect-or-misleading responses."
        ),
        "post_calibration_result": "as_006 correctly routes to BORDERLINE"
    },
    "alignment_improvement": {
        "pre_calibration": f"{pre_cal:.1%}",
        "post_calibration": f"{post_cal:.1%}",
        "improvement": f"{(post_cal - pre_cal):.1%}"
    },
    "routing_summary": {
        "quality_layer_PASS": pass_ids,
        "human_review_BORDERLINE": border_ids,
        "governance_layer_FAIL": fail_ids
    },
    "per_sample_results": calibrated_results,
    "artifact_note": (
        "All aspect verdicts bound to SHA-256 input hashes. "
        "Self-issued: proves non-alteration, not independent recomputability. "
        "Source: Federico Blanco Sanchez-Llanos, "
        "Enforcement Infrastructure Capital and Compute. "
        "Acknowledged in Phase 4b: this requirement emerged from the "
        "LinkedIn exchange and is built independently using Python hashlib."
    ),
    "langfuse": {
        "summary_trace": summary_trace["langfuse_id"],
        "production_value": (
            "In live mode: traces persist across sessions, cross-phase "
            "alignment comparison available in dashboard, Phase 6 "
            "regression alarm queries these scores directly."
        )
    }
}

output_path = DRIVE_PATH + "phase04b_judge_alignment_results.json"
with open(output_path, "w") as f:
    json.dump(output_4b, f, indent=2)

print(f"Results saved: {output_path}")
print()
print("Summary:")
print(f"  Aspect version v1 (5 aspects): {pre_cal:.1%} alignment")
print(f"  Aspect version v2 (6 aspects): {post_cal:.1%} alignment")
print(f"  Improvement:                   {(post_cal - pre_cal):.1%}")
print()
print("Routing (post-calibration):")
print(f"  PASS      -> quality layer:    {pass_ids}")
print(f"  BORDERLINE -> human review:    {border_ids}")
print(f"  FAIL      -> governance layer: {fail_ids}")
print()
print("Both aspect versions and full calibration history saved.")
print("Phase 6 regression alarm will query phase_04b_post_calibration_alignment.")

Results saved: /content/drive/MyDrive/python-ai-governance-p2/data/phase04b_judge_alignment_results.json

Summary:
  Aspect version v1 (5 aspects): 83.3% alignment
  Aspect version v2 (6 aspects): 100.0% alignment
  Improvement:                   16.7%

Routing (post-calibration):
  PASS      -> quality layer:    ['as_001', 'as_002', 'as_003']
  BORDERLINE -> human review:    ['as_006']
  FAIL      -> governance layer: ['as_004', 'as_005']

Both aspect versions and full calibration history saved.
Phase 6 regression alarm will query phase_04b_post_calibration_alignment.


## Phase 4b Findings: Judge Alignment: Calibration and Post-Calibration Score

**Judge model:** claude-sonnet-4-6 (cross-model, independent throughout)

**What was built:** A calibration workflow addressing the Phase 4a
disagreement between the Claude AspectCritic judge and a human reviewer
on sample as_006. A sixth aspect (coverage_completeness) was added to
the rubric, the routing logic was updated to distinguish completeness
failures from accuracy failures, and the full six-sample alignment
evaluation was re-run.

**What was found:**

| Sample | Name                         | Human      | v1 Judge | v2 Judge   |
|--------|------------------------------|------------|----------|------------|
| as_001 | Article 14 clean response    | PASS       | PASS ✓   | PASS ✓     |
| as_002 | Article 10 clean response    | PASS       | PASS ✓   | PASS ✓     |
| as_003 | Article 99 penalty query     | PASS       | PASS ✓   | PASS ✓     |
| as_004 | Hallucination: wrong figures | FAIL       | FAIL ✓   | FAIL ✓     |
| as_005 | Oversight misrepresentation  | FAIL       | FAIL ✓   | FAIL ✓     |
| as_006 | NIST GOVERN partial response | BORDERLINE | PASS ✗   | BORDERLINE ✓|

Pre-calibration alignment (5 aspects):  5/6 = 83.3%
Post-calibration alignment (6 aspects): 6/6 = 100.0%
Improvement: 16.7%

**The calibration finding:** The disagreement was not a judge error.
The v1 five-aspect rubric was genuinely incomplete: none of the five
aspects captured whether a response was actionably complete for a
deployer. Adding coverage_completeness as a sixth aspect resolved the
disagreement without changing how the five existing aspects evaluate
the same samples. The routing logic correctly distinguishes a
completeness failure (BORDERLINE, routes to human review) from an
accuracy failure (FAIL, routes to governance block).

**The routing distinction that matters:**
- as_004 and as_005: accuracy and compliance failures. FAIL.
  Routes to governance layer. Merge blocked.
- as_006: coverage failure only. BORDERLINE.
  Routes to human review. A human reviewer checks whether the
  incomplete response is sufficient for the specific deployer context.
  The reviewer may pass it or request a more complete retrieval.
  This is the correct outcome: incomplete-but-accurate is a different
  problem from incorrect-or-misleading.

**The signed artifact note:** This requirement emerged from the LinkedIn
exchange with Federico Blanco Sanchez-Llanos and is built independently
using Python hashlib rather than any external vendor endpoint. Every
aspect verdict in both v1 and v2 is bound to a SHA-256 hash of the
specific inputs. Self-issued hashes prove non-alteration. They do not
prove independent recomputability. Both properties are documented
explicitly and will be named in the Phase 7 synthesis.

**Langfuse note:** In simulated mode Langfuse traces are constructed
locally and not transmitted. In live mode they persist across sessions,
enable cross-phase alignment comparison in a single dashboard, and
provide the score store that Phase 6's regression alarm queries directly.
The trace structure built here is identical in both modes.

**Next step:** Phase 5a (05a_promptfoo_owasp_llm.ipynb) begins the
red-teaming layer: Promptfoo with the OWASP LLM Top 10 (2025) preset,
running against the baseline RAG pipeline established in Phase 1.